# 01 — Data exploration

Quick look at the raw datasets: counts, class balance, and a small sample grid per category.
Run this *after* you've downloaded EuroSAT / xBD into `data/raw/`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from cv_diffusion.utils.io import list_images

DATA_ROOT = Path('../data/raw')
print('Contents of data/raw:', sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))

In [ ]:
# Per-class counts in the EuroSAT folder (if available).
eurosat = DATA_ROOT / 'eurosat'
if eurosat.exists():
    counts = {p.name: len(list_images(p)) for p in eurosat.iterdir() if p.is_dir()}
    print(counts)
else:
    print('EuroSAT not found — run scripts/download_data.sh eurosat first.')

In [ ]:
# Show a 3x3 sample from the first available class.
if eurosat.exists():
    first_class = next(p for p in eurosat.iterdir() if p.is_dir())
    paths = list_images(first_class)[:9]
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for ax, p in zip(axes.flat, paths):
        ax.imshow(Image.open(p))
        ax.set_axis_off()
    plt.suptitle(first_class.name)
    plt.show()

## Make a stratified train/val/test split for xBD

Run only once. Produces `data/processed/xbd_classifier/{train,val,test}/<class>/...` using symlinks so it's cheap.

In [ ]:
import random, shutil
random.seed(42)
RAW = Path('../data/processed/xbd_tiles')
DEST = Path('../data/processed/xbd_classifier')
splits = {'train': 0.7, 'val': 0.15, 'test': 0.15}

if RAW.exists():
    for cls_dir in [p for p in RAW.iterdir() if p.is_dir()]:
        images = sorted(list_images(cls_dir))
        random.shuffle(images)
        n = len(images)
        n_train = int(n * splits['train'])
        n_val = int(n * splits['val'])
        chunks = {
            'train': images[:n_train],
            'val': images[n_train:n_train+n_val],
            'test': images[n_train+n_val:],
        }
        for split, paths in chunks.items():
            out_dir = DEST / split / cls_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            for src in paths:
                target = out_dir / src.name
                if target.exists():
                    continue
                try:
                    target.symlink_to(src.resolve())
                except OSError:
                    shutil.copy2(src, target)
    print('Splits created at', DEST)
else:
    print('Run cv-preprocess first to create xbd_tiles.')